In [1]:
## Imports

import pandas as pd
import numpy as np
import datetime as dt

import yfinance as yf

from fredapi import Fred

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


In [2]:
# FRED API
fred = Fred(api_key='09d192ed770bc9badceb48bb9a3cc664')
#fred = Fred(api_key="YOUR_API_KEY_HERE")


# this for pulling the treasury yields: 2 and 10 year (could add more later)
# https://fred.stlouisfed.org/series/DGS2
# https://fred.stlouisfed.org/series/DGS10)
series_ids = {
    "DGS10": "10_Year_Treasury_Yield",
    "DGS2": "2_Year_Treasury_Yield"
}

fred_data = {}

for series, name in series_ids.items():
    data = fred.get_series(series)
    fred_data[name] = data

fred_df = pd.DataFrame(fred_data)
fred_df.index = pd.to_datetime(fred_df.index)
fred_df.sort_index(inplace=True)

# Test Yfinance

global_indices = [
    "^GSPC",        # S&P 500 (US Large Cap)
    "^DJI",         # Dow Jones Industrial Average (US)
    "^NDX",         # Nasdaq 100
    "^HSI",         # Hang Seng (Hong Kong)
    "^N225",        # Nikkei 225 (Japan)
    "000001.SS",    # Shanghai Composite (China)
    "^STOXX50E",    # Euro Stoxx 50 (Europe)
    "^FTSE",        # FTSE 100 (UK)
]

'''financials = [
    "JPM",   # JPMorgan Chase
    "BAC",   # Bank of America
    "WFC",   # Wells Fargo
]

tech = [
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "NVDA",  # Nvidia
    "TSLA",  #Tesla
]'''

energy = [
    "XOM",   # Exxon Mobil
    "CVX",   # Chevron
    "SLB",   # Schlumberger
]

sector_etfs = [
    "XLF",  # Financials
    "XLK",  # Technology
    "XLE",  # Energy
    "XLY",  # Consumer Discretionary
    "XLP",  # Consumer Staples
    "XLV",  # Healthcare
    "XLI",  # Industrials
    "XLB",  # Materials
    "XLU",  # Utilities
    "IYR",  # Real Estate
]

commodities = [
    "GLD",     # Gold
    "SLV",     # Silver
    "USO",     # Crude Oil
]

usd = ['DX-Y.NYB']

vol = ["^VIX"]

tickers = (
    global_indices
   # + financials
   # + tech
    + energy
    + sector_etfs
    + commodities
    + vol
    + usd
)


# set start date as of 1995 onwards
start_date = '1995-01-01'
end_date = dt.datetime.now().strftime('%Y-%m-%d')

market_df = yf.download(
    tickers,
    start = start_date,
    end = end_date,
    interval = '1wk',
)


adj_close = market_df['Close'].copy()

fred_weekly = fred_df.resample("W-FRI").last()
adj_close_weekly = adj_close.resample('W-FRI').last()
combine_sc = adj_close_weekly.join(fred_weekly, how = "inner")


combine_sc = combine_sc.dropna(how="all", axis=1)
combine_sc = combine_sc.dropna(how="all", axis=0)

'''ticker_map = {
    # Global indices
    "^GSPC": "SP500",
    "^DJI": "DOW_JONES",
    "^NDX": "NASDAQ100",
    "^HSI": "HANG_SENG",
    "^N225": "NIKKEI_225",
    "000001.SS": "SHANGHAI_COMPOSITE",
    "^STOXX50E": "EURO_STOXX_50",
    "^FTSE": "FTSE_100",

    # Financial stocks
    "JPM": "JPMORGAN",
    "BAC": "BANK_OF_AMERICA",
    "WFC": "WELLS_FARGO",

    # Tech stocks
    "AAPL": "APPLE",
    "MSFT": "MICROSOFT",
    "NVDA": "NVIDIA",
    "TSLA": "TESLA",

    # Energy stocks
    "XOM": "EXXON_MOBIL",
    "CVX": "CHEVRON",
    "SLB": "SCHLUMBERGER",

    # Sector ETFs
    "XLF": "FINANCIALS_XLF",
    "XLK": "TECH_XLK",
    "XLE": "ENERGY_XLE",
    "XLY": "CONSUMER_DISCRETIONARY_XLY",
    "XLP": "CONSUMER_STAPLES_XLP",
    "XLV": "HEALTHCARE_XLV",
    "XLI": "INDUSTRIALS_XLI",
    "XLB": "MATERIALS_XLB",
    "XLU": "UTILITIES_XLU",
    "IYR": "REAL_ESTATE_IYR",

    # Commodities
    "GLD": "GOLD_GLD",
    "SLV": "SILVER_SLV",
    "USO": "CRUDE_OIL_USO",

    # Volatility index
    "^VIX": "VIX",

    # FRED data already readable but you can rename if you want
    "10_Year_Treasury_Yield": "TREASURY_10Y",
    "2_Year_Treasury_Yield": "TREASURY_2Y",
}
'''
'''combine_sc = combine_sc.rename(columns=ticker_map)'''

combine_sc.head()

[*********************100%***********************]  26 of 26 completed


,000001.SS,CVX,DX-Y.NYB,GLD,IYR,SLB,SLV,USO,XLB,XLE,...,^DJI,^FTSE,^GSPC,^HSI,^N225,^NDX,^STOXX50E,^VIX,10_Year_Treasury_Yield,2_Year_Treasury_Yield
1995-01-06,NaN,7.065623,89.610001,NaN,NaN,6.349371,NaN,NaN,NaN,NaN,...,3867.409912,3065.000000,460.679993,7683.299805,19519.460938,401.589996,NaN,13.13,7.87,7.64
1995-01-13,NaN,7.045663,88.139999,NaN,NaN,6.241495,NaN,NaN,NaN,NaN,...,3908.459961,3048.300049,465.970001,7252.299805,19331.169922,410.480011,NaN,11.10,7.69,7.39
1995-01-20,NaN,7.384970,87.449997,NaN,NaN,6.549718,NaN,NaN,NaN,NaN,...,3869.429932,2995.000000,464.779999,7278.100098,18840.220703,410.130005,NaN,12.15,7.82,7.50
1995-01-27,NaN,7.384970,87.620003,NaN,NaN,6.673005,NaN,NaN,NaN,NaN,...,3857.989990,3022.199951,470.390015,7297.100098,18104.349609,407.220001,NaN,11.25,7.66,7.29
1995-02-03,NaN,7.305132,88.190002,NaN,NaN,6.642186,NaN,NaN,NaN,NaN,...,3928.639893,3059.699951,478.649994,7478.899902,18538.970703,416.140015,NaN,10.98,7.49,7.14


In [3]:

#combine_sc.to_csv("combined_weekly_dataset.csv")

In [4]:
#PHUC: This is the start of my part

#Download historical market data
#Download macroeconomic data (Treasury yields) from FRED
#Convert all data to weekly frequency
#Merge market and macro data into one dataset
#Save prepared dataset for later steps

reg = ["SPY","QQQ","TLT","GLD","^VIX"]
etf = ["XLF","XLK","XLE","XLY","XLP","XLV","XLI","XLB","XLU","IYR"]
tickers = reg + etf

market_df = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=True,
    progress=False
)

market_df = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=True
)

close = market_df["Close"].copy()
close_weekly = close.resample("W-FRI").last()
fred_weekly = fred_df.resample("W-FRI").last().ffill()
combine_phuc = close_weekly.join(fred_weekly, how="inner")
combine_phuc = combine_phuc.rename(columns={"^VIX": "VIX"})
combine_phuc = combine_phuc.dropna(how="all", axis=1)
combine_phuc = combine_phuc.dropna(how="all", axis=0)
#print(list(combine_phuc.columns))
combine_phuc.head()
#combine_phuc.to_parquet("data_prep.parquet")


[*********************100%***********************]  15 of 15 completed


,GLD,IYR,QQQ,SPY,TLT,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,VIX,10_Year_Treasury_Yield,2_Year_Treasury_Yield
1995-01-06,NaN,NaN,NaN,26.665783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.13,7.87,7.64
1995-01-13,NaN,NaN,NaN,27.063885,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.10,7.69,7.39
1995-01-20,NaN,NaN,NaN,26.955320,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.15,7.82,7.50
1995-01-27,NaN,NaN,NaN,27.281054,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.25,7.66,7.29
1995-02-03,NaN,NaN,NaN,27.814932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.98,7.49,7.14


In [5]:
## remove overlapping before mapping to readable.

overlap = combine_sc.columns.intersection(combine_phuc.columns)
print("Duplicate columns:", list(overlap))

Duplicate columns: ['GLD', 'IYR', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', '10_Year_Treasury_Yield', '2_Year_Treasury_Yield']


In [6]:
combine_phuc = combine_phuc.drop(columns=overlap, errors="ignore")

In [7]:
merge_combine = combine_sc.merge(
    combine_phuc,
    left_index=True,
    right_index=True,
    how="inner"
)

In [8]:
merge_combine = merge_combine.drop(columns=["VIX"], errors="ignore")

In [9]:
print(merge_combine.columns)

Index(['000001.SS', 'CVX', 'DX-Y.NYB', 'GLD', 'IYR', 'SLB', 'SLV', 'USO',
       'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', 'XOM',
       '^DJI', '^FTSE', '^GSPC', '^HSI', '^N225', '^NDX', '^STOXX50E', '^VIX',
       '10_Year_Treasury_Yield', '2_Year_Treasury_Yield', 'QQQ', 'SPY', 'TLT'],
      dtype='object')


In [10]:
ticker_map = {
    # Global indices
    "^GSPC": "SP500",
    "^DJI": "DOW_JONES",
    "^NDX": "NASDAQ100",
    "^HSI": "HANG_SENG",
    "^N225": "NIKKEI_225",
    "000001.SS": "SHANGHAI_COMPOSITE",
    "^STOXX50E": "EURO_STOXX_50",
    "^FTSE": "FTSE_100",

    # Market ETFs
    "SPY": "SP500_SPY",
    "QQQ": "NASDAQ100_QQQ",
    "TLT": "TREASURY_BOND_TLT",

    # Sector ETFs
    "XLF": "FINANCIALS_XLF",
    "XLK": "TECH_XLK",
    "XLE": "ENERGY_XLE",
    "XLY": "CONSUMER_DISCRETIONARY_XLY",
    "XLP": "CONSUMER_STAPLES_XLP",
    "XLV": "HEALTHCARE_XLV",
    "XLI": "INDUSTRIALS_XLI",
    "XLB": "MATERIALS_XLB",
    "XLU": "UTILITIES_XLU",
    "IYR": "REAL_ESTATE_IYR",

    # Commodities
    "GLD": "GOLD_GLD",
    "SLV": "SILVER_SLV",
    "USO": "CRUDE_OIL_USO",

    # Volatility index
    "^VIX": "VIX",

    # FRED data already readable but you can rename if you want
    "10_Year_Treasury_Yield": "TREASURY_10Y",
    "2_Year_Treasury_Yield": "TREASURY_2Y",

    "DX-Y.NYB" : "usd_index"
}

merge_combine = merge_combine.rename(columns=ticker_map)


In [11]:
merge_combine

,SHANGHAI_COMPOSITE,CVX,usd_index,GOLD_GLD,REAL_ESTATE_IYR,SLB,SILVER_SLV,CRUDE_OIL_USO,MATERIALS_XLB,ENERGY_XLE,...,HANG_SENG,NIKKEI_225,NASDAQ100,EURO_STOXX_50,VIX,TREASURY_10Y,TREASURY_2Y,NASDAQ100_QQQ,SP500_SPY,TREASURY_BOND_TLT
1995-01-06,NaN,7.065623,89.610001,NaN,NaN,6.349371,NaN,NaN,NaN,NaN,...,7683.299805,19519.460938,401.589996,NaN,13.130000,7.87,7.64,NaN,26.665783,NaN
1995-01-13,NaN,7.045663,88.139999,NaN,NaN,6.241495,NaN,NaN,NaN,NaN,...,7252.299805,19331.169922,410.480011,NaN,11.100000,7.69,7.39,NaN,27.063885,NaN
1995-01-20,NaN,7.384970,87.449997,NaN,NaN,6.549718,NaN,NaN,NaN,NaN,...,7278.100098,18840.220703,410.130005,NaN,12.150000,7.82,7.50,NaN,26.955320,NaN
1995-01-27,NaN,7.384970,87.620003,NaN,NaN,6.673005,NaN,NaN,NaN,NaN,...,7297.100098,18104.349609,407.220001,NaN,11.250000,7.66,7.29,NaN,27.281054,NaN
1995-02-03,NaN,7.305132,88.190002,NaN,NaN,6.642186,NaN,NaN,NaN,NaN,...,7478.899902,18538.970703,416.140015,NaN,10.980000,7.49,7.14,NaN,27.814932,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-06,4124.193848,189.940002,98.989998,473.510010,98.742844,46.900002,75.940002,108.769997,49.639248,56.202785,...,25757.289062,55620.839844,24643.019531,5719.899902,29.490000,4.15,3.56,598.994690,670.548706,88.107956
2026-03-13,4095.447998,196.820007,100.360001,460.839996,97.227097,44.720001,72.690002,119.889999,48.972214,57.325451,...,25465.599609,53819.609375,24380.730469,5716.609863,27.190001,4.28,3.73,592.972290,660.486206,86.195602
2026-03-20,3957.052979,201.729996,99.650002,413.380005,93.188431,46.630001,61.520000,121.430000,46.771999,58.925003,...,25277.320312,53372.531250,23898.150391,5501.279785,26.780001,4.39,3.88,581.326965,648.570007,85.488426
2026-03-27,3913.724121,211.149994,100.150002,414.700012,92.610001,53.500000,63.439999,124.199997,48.693455,62.153904,...,24951.880859,53373.070312,23132.769531,5505.799805,31.049999,4.44,3.88,562.580017,634.090027,85.299179


In [12]:
merge_combine.to_parquet("combined_weekly_dataset.parquet")

In [13]:
merge_combine.columns

Index(['SHANGHAI_COMPOSITE', 'CVX', 'usd_index', 'GOLD_GLD', 'REAL_ESTATE_IYR',
       'SLB', 'SILVER_SLV', 'CRUDE_OIL_USO', 'MATERIALS_XLB', 'ENERGY_XLE',
       'FINANCIALS_XLF', 'INDUSTRIALS_XLI', 'TECH_XLK', 'CONSUMER_STAPLES_XLP',
       'UTILITIES_XLU', 'HEALTHCARE_XLV', 'CONSUMER_DISCRETIONARY_XLY', 'XOM',
       'DOW_JONES', 'FTSE_100', 'SP500', 'HANG_SENG', 'NIKKEI_225',
       'NASDAQ100', 'EURO_STOXX_50', 'VIX', 'TREASURY_10Y', 'TREASURY_2Y',
       'NASDAQ100_QQQ', 'SP500_SPY', 'TREASURY_BOND_TLT'],
      dtype='object')

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=059a5e93-4459-411b-9a58-2a578fd7a892' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>